### frontend checkup

In [1]:
pre_ans = """Hi, my name is rushal and i love to tell you that i love football"""

In [2]:
qno = """tell me about your self"""

In [3]:
def length_check(text: str) -> bool:
    words = text.strip().split()
    if len(words) < 5 or len(text) < 15:
        return False  # reject
    return True

In [4]:
import re

NON_ANSWERS = {
    "idk", "i don't know", "no idea", "skip", "n/a", "pass", "none", "nothing"
}

REGEX_PATTERNS = [
    r"^(i\s*don'?t\s*know)$",
    r"^(no\s*idea)$",
    r"^(skip|pass|next)$",
    r"^\s*n/?a\s*$",
]

def non_answer_check(text: str) -> bool:
    normalized = text.strip().lower()
    if normalized in NON_ANSWERS:
        return False
    for pattern in REGEX_PATTERNS:
        if re.match(pattern, normalized):
            return False
    return True

In [32]:
import math
from collections import Counter

def shannon_entropy(text: str) -> float:
    if not text:
        return 0.0
    freq = Counter(text)
    length = len(text)
    return -sum((c/length) * math.log2(c/length) for c in freq.values())

def gibberish_check(text: str) -> bool:
    text = text.strip()
    entropy = shannon_entropy(text.lower())
    
    # English sentences typically: 3.5 – 4.5
    # "asdfghjkl" => very high entropy (~3.1 but low vowel ratio)
    # "aaaaaaaa" => very low entropy (~0)
    
    vowels = sum(1 for c in text.lower() if c in "aeiou")
    vowel_ratio = vowels / max(len(text), 1)
    print("entropy",entropy)
    print("vowel ratio : ",vowel_ratio)
    if entropy < 1.5:      # too repetitive
        return False
    if entropy > 4.8:      # too random
        return False
    if vowel_ratio < 0.15: # no vowels → likely mash
        return False
    return True

In [33]:
def precheck(user_input: str) -> tuple[bool, str]:
    if not gibberish_check(user_input):
        return False, "gibberish"
    if not non_answer_check(user_input):
        return False, "non_answer"
    if not length_check(user_input):
        return False, "too_short"
    return True, "ok"

In [40]:
ans = """sdfsdgsadhci"""

In [41]:
# Usage
ok, reason = precheck(ans)
if not ok:
    print({"error": reason})   # skip LLM entirely — save money + latency
else:
    print({"good":reason})
# else → proceed to deterministic metrics → then LLM judge

entropy 2.792481250360578
vowel ratio :  0.16666666666666666
{'error': 'too_short'}


### LETS SEE WHAT EMBEDING IS CAPABLE OF

# IMPORTS

In [1]:
import ollama
from sklearn.metrics.pairwise import cosine_similarity
import json
# for graph rag
import networkx as nx

#for CRAG with search if irrelevent
from ddgs import DDGS
import torch
from TTS.api import TTS

d:\Rushal\VS_code\Python\LocalAis\OLLAMA\ProjectIII\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'is_torch_greater_or_equal' from 'transformers.utils.import_utils' (d:\Rushal\VS_code\Python\LocalAis\OLLAMA\ProjectIII\Lib\site-packages\transformers\utils\import_utils.py)

In [ ]:
def get_embedding(text,model):
    response = ollama.embeddings(model=model, prompt=text)
    return response['embedding']

In [5]:
txt1="Its better if you stay here today"
txt2="better"

In [6]:
m1="all-minilm:latest"
m2="nomic-embed-text:latest"

In [10]:
score1 = cosine_similarity([get_embedding(txt1,model=m1)], [get_embedding(txt2,model=m1)])
score2 = cosine_similarity([get_embedding(txt1,model=m2)], [get_embedding(txt2,model=m2)])

In [11]:
score1,score2

(array([[0.33729827]]), array([[0.59257481]]))

## Graph RAG

### What is it?
GraphRAG combines standard vector retrieval with a Knowledge Graph. Instead of searching for context purely by semantic text similarity, it extracts entities (e.g., People, Tools, Organizations) and their relationships (e.g., USES, DEPENDS_ON) to query graph databases.

### When is it mostly used?
Complex datasets where facts are interconnected across multiple documents (e.g., legal contracts, medical records, financial supply chains).

Answering global summaries across an entire dataset (e.g., "What are the key themes and interconnected risks in all financial reports?").

### World Popularity & Recent Trends
GraphRAG gained massive traction following research and open-source tooling from Microsoft Research. Combining graph community detection (like Leiden algorithms) with vector search is currently one of the highest-performing enterprise patterns.

### Real-Life Application
Investigating corporate fraud: connecting shell companies, directors, and bank transactions across thousands of leaked PDF documents.

### Most Important Thing to Know
Graph extraction using LLMs during indexing is computationally expensive. Every sentence or paragraph must be scanned by an LLM to extract entities and relations.

In [12]:
# 1. Initialize local graph database
G = nx.DiGraph()

# Sample knowledge extraction prompt
def extract_triplets(text):
    prompt = f"""Extract core (subject, relation, object) knowledge triplets from this text.
Return ONLY a raw JSON array of objects with keys "subject", "relation", "object".
Text: {text}"""
    
    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
        format='json'
    )
    return json.loads(response['message']['content'])

In [29]:
# Build simple graph from text
sample_text = "Llama3.2 is an open model developed by Meta. Ollama runs local models including Llama3.2."
sample_text = sample_text.lower()
try:
    triplets = extract_triplets(sample_text)
    print("triplets : ",triplets)
    print("triplets", triplets)
    for t in triplets:
        print("t in triplets : ",t['subject'])
        print("t in triplets : ",t['object'])
        print("t in triplets : ",t['relation'])
        G.add_edge(t['subject'], t['object'], relation=t['relation'])
except Exception:
    # Fallback structure for demonstration
    G.add_edge("Llama3.2", "Meta", relation="developed_by")
    G.add_edge("Ollama", "Llama3.2", relation="runs")

triplets :  {'subject': 'llama3.2', 'relation': 'is', 'object': 'an open model'}
triplets {'subject': 'llama3.2', 'relation': 'is', 'object': 'an open model'}


In [ ]:
# Graph Traverse Query function
def graph_rag_query(entity):
    if entity in G:
        neighbors = list(G.neighbors(entity))
        edges = [f"{entity} -> {G[entity][nbr]['relation']} -> {nbr}" for nbr in neighbors]
        return "\n".join(edges)
    return "No graph entity found."

context = graph_rag_query("Llama3.2")
print(f"Retrieved Graph Context:\n{context}")

Retrieved Graph Context:
No graph entity found.


In [87]:
list(G.neighbors("Ollama"))

['Llama3.2']

In [25]:
if "ollama" in G:
    print('yes')
else:
    print('no')

no


## Agentic RAG
### What is it?
Agentic RAG transforms retrieval into a multi-step tool-calling reasoning loop. Instead of executing a single static lookup, an LLM agent breaks down the query, decides which search tools or databases to invoke, evaluates intermediate answers, and iteratively retrieves missing details.

### When is it mostly used?
Multi-hop user queries requiring multi-step reasoning (e.g., "Compare the Q3 revenue of Company A with the R&D budget of Company B").

Dynamic environments with multiple data sources (e.g., Vector DBs, SQL databases, Web APIs).

### World Popularity & Recent Trends
Agentic architectures (built with frameworks like LangGraph or LlamaIndex Agents) have become the dominant standard for building production-grade enterprise assistants.

### Real-Life Application
An automated customer support agent that checks order status in a SQL database, retrieves return policy rules from a Vector DB, and issues a refund ticket via API.

### Most Important Thing to Know
Agent loops increase system latency and API token usage. Strict termination logic and guardrails are required to prevent infinite agentic loops.

In [49]:
def search_vector_db(query):
    return f"[Vector DB Result]: Found documentation for query '{query}'."

def query_sql_db(query):
    return f"[SQL Result]: Customer 101 spent $450 in October."

# Agentic Tool Routing Logic
def agentic_rag(user_query):
    system_prompt = """You are a smart router agent. Determine which tool to call based on user intent.
Available tools:
1. "vector_db" for general documentation or text search.
2. "sql_db" for structured numerical user data or transaction queries.

Return ONLY a valid JSON: {"tool": "vector_db" | "sql_db", "search_term": "<query>"}
"""

    response = ollama.chat(
        model='llama3.2',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_query}
        ],
        format='json'
    )
    
    decision = json.loads(response['message']['content'])
    print("decision : ",decision)
    
    # Execute selected tool dynamically
    if decision.get("tool") == "sql_db":
        retrieved_context = query_sql_db(decision.get("search_term"))
    else:
        retrieved_context = search_vector_db(decision.get("search_term"))
        
    # Final Answer Generation
    final_response = ollama.chat(
        model='llama3.2',
        messages=[
            {'role': 'system', 'content': f"Answer using context: {retrieved_context}"},
            {'role': 'user', 'content': user_query}
        ]
    )
    return final_response['message']['content']

In [50]:
print(agentic_rag("How much money did customer 101 spend recently?"))

decision :  {'tool': 'sql_db', 'search_term': 'customer_id=101 spend_recently'}
Unfortunately, I don't have the specific information on what time frame you're referring to as "recently". If you provide more details or clarify what you mean by "recently", I'd be happy to try and help. Was it within October, November, or something else?


1. can't we use the attribute named tools?

## Self-RAG & Corrective RAG (CRAG)

### What is it?
Self-RAG: The LLM actively reflects on its own outputs by generating internal tags to judge context relevance, output support, and hallucination risks.

Corrective RAG (CRAG): Introduces a light evaluator step right after document retrieval. *If retrieved vector documents are evaluated as irrelevant or low-confidence, CRAG triggers a web search (e.g., Tavily or DuckDuckGo)* or query-rewriting workflow to correct the pipeline before generation.

### When is it mostly used?
High-accuracy environments (medical, legal, financial) where hallucinated facts or poor vector matches cause severe failure.

Systems operating over incomplete or noisy vector stores.

### World Popularity & Recent Trends
CRAG and Self-RAG form the core framework behind state-of-the-art reasoning search engines (e.g., Perplexity AI). They ensure high accuracy by preventing low-quality contexts from being passed to generation steps.

### Real-Life Application
An enterprise research engine that dynamically rewrites ambiguous internal user queries and falls back to live Google/web searches if internal documents yield insufficient answers.

### Most Important Thing to Know
Self-evaluation checks significantly increase inference latency. To optimize performance, lightweight models (like Llama 3.2 3B or fine-tuned classifiers) are typically used for evaluation stages, reserving larger models for final synthesis.

In [51]:
def dummy_vector_search(query):
    # Simulating a bad or irrelevant retrieval
    return "Unrelated topic text about cooking recipes."

def fallback_web_search(query):
    return f"[Web Search Result]: Latest factual search data for '{query}'."

def evaluate_relevance(query, retrieved_doc):
    prompt = f"""Evaluate if the retrieved document is RELEVANT to answer the query.
Query: {query}
Document: {retrieved_doc}

Return ONLY JSON: {{"is_relevant": true | false}}"""

    res = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
        format='json'
    )
    return json.loads(res['message']['content']).get("is_relevant", False)

In [52]:
# CRAG Pipeline Execution
query = "What is the latest release date for Llama 3.2?"
doc = dummy_vector_search(query)

# Evaluate retrieval relevance
is_valid = evaluate_relevance(query, doc)

if is_valid:
    final_context = doc
    print("--- CRAG Status: Vector DB Retrieval Relevant ---")
else:
    print("--- CRAG Status: Low Relevance Detected! Triggering Web Search Fallback ---")
    final_context = fallback_web_search(query)

# Generate final answer
answer = ollama.chat(
    model='llama3.2',
    messages=[
        {'role': 'system', 'content': f"Context: {final_context}"},
        {'role': 'user', 'content': query}
    ]
)
print(f"Final Answer:\n{answer['message']['content']}")

--- CRAG Status: Low Relevance Detected! Triggering Web Search Fallback ---
Final Answer:
I don't have information on a release date for Llama 3.2. My training data may not be up-to-date or might not include this specific information. Can I help you with something else?


In [64]:
# 1. Query Rewriter: Converts user questions into optimal search engine keywords
def rewrite_query_for_search(user_query: str) -> str:
    prompt = f"""You are a search query optimizer. Convert the user's question into 2-4 concise search engine keywords.
User Question: {user_query}

Return ONLY a JSON object: {{"search_query": "<keywords>"}}"""

    try:
        res = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': prompt}],
            format='json'
        )
        return json.loads(res['message']['content']).get("search_query", user_query)
    except Exception:
        return user_query

# 2. Web Search Tool with Clean Snippet Parsing
def web_search_fallback(query: str) -> str:
    optimized_keywords = rewrite_query_for_search(query)
    print(f"\n[CRAG Action]: Rewrote query to '{optimized_keywords}' and searching DuckDuckGo...")
    
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(optimized_keywords, max_results=4):
            results.append(f"Source: {r.get('title')}\nSnippet: {r.get('body')}")
            
    return "\n\n".join(results) if results else "No relevant web information found."

# 3. Evaluator: Checks if retrieved text is relevant
def evaluate_relevance(query: str, retrieved_doc: str) -> bool:
    eval_prompt = f"""Determine if the context contains relevant details to answer the query.
Query: {query}
Context: {retrieved_doc}

Return ONLY JSON: {{"is_relevant": true | false}}"""

    try:
        res = ollama.chat(
            model='llama3.2',
            messages=[{'role': 'user', 'content': eval_prompt}],
            format='json'
        )
        return json.loads(res['message']['content']).get("is_relevant", False)
    except Exception:
        return False

# 4. Main CRAG Flow
def run_crag(user_query: str, mock_vector_doc: str):
    print(f"User Query: {user_query}")
    
    # Evaluate initial vector retrieval
    if evaluate_relevance(user_query, mock_vector_doc):
        print("[CRAG Evaluation]: Vector store context is RELEVANT.")
        final_context = mock_vector_doc
    else:
        print("[CRAG Evaluation]: Vector store context is IRRELEVANT! Fallback triggered.")
        final_context = web_search_fallback(user_query)

    # Prompt forcing the model to rely STRICTLY on the retrieved search context
    system_prompt = f"""You are a precise factual assistant. Answer the question using ONLY the provided search context below. 
Do NOT rely on internal training cutoffs. If the facts are present in the search context, synthesize them clearly.

Search Context:
{final_context}"""

    response = ollama.chat(
        model='llama3.2',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_query}
        ]
    )
    return response['message']['content']

if __name__ == "__main__":
    query = "What date is today in NEPALI date with day name"
    bad_vector_context = "Italian pasta recipes and dough making techniques."
    
    answer = run_crag(query, bad_vector_context)
    print("\n" + "="*40 + "\nFINAL ANSWER:\n" + "="*40)
    print(answer)

User Query: What date is today in NEPALI date with day name
[CRAG Evaluation]: Vector store context is IRRELEVANT! Fallback triggered.

[CRAG Action]: Rewrote query to 'nepali calendar date today' and searching DuckDuckGo...

FINAL ANSWER:
Unfortunately, I don't have real-time information on the current date in Nepali. However, based on the search context provided, I can suggest that you check the "Nepali Date Today" website or the "Nepcalendar" online calendar for the most up-to-date and accurate information.

If you would like to know the Nepali date of a specific day, I can try to provide you with general information. However, please note that I won't be able to provide the exact date without access to real-time information.

That being said, according to the search context provided, the current month is Asoj (also known as Ashwin) in the Nepali calendar. If you provide me with the specific day of the week or any other relevant details, I can try to help you determine the correct da

## Evaluation technique

In [ ]:
# Naive Bayes Classifier with TF-IDF Vectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

# 1. Training data (Intent labels)
train_texts = [
    "I don't know the answer", "skip this question", "no idea", "pass",
    "Overfitting happens when a model learns noise", "It stores training data variance",
    "",
    "What is your favorite food?", "Who won the football match?"
]
train_labels = ["NON_ANSWER", "NON_ANSWER", "NON_ANSWER", "NON_ANSWER",
                "VALID_ANSWER", "VALID_ANSWER", 
                "VALID_ANSWER",
                "OFF_TOPIC", "OFF_TOPIC"]

In [20]:
# 2. Pipeline: TF-IDF -> Naive Bayes
model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2)), MultinomialNB())
model.fit(train_texts, train_labels)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('multinomialnb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[<U12](3,)","['NON_ANSWER','OFF_TOPIC','VALID_ANSWER']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None


In [25]:
# 3. Predict candidate intent (< 1ms CPU)
candidate_reply = "myself rushal"
predicted_intent = model.predict([candidate_reply])[0]
probabilities = model.predict_proba([candidate_reply])[0]

print(f"Predicted Intent: {predicted_intent}, {probabilities}")

Predicted Intent: NON_ANSWER, [0.44444444 0.22222222 0.33333333]
